In [ ]:
import numpy as np
import pandas as pd

#df = pd.read_parquet("s3://tradebot-config-tokyo/data/trade/APTUSDT/2024-01.parquet")
#df = pd.read_parquet("s3://tradebot-config-tokyo/data/book/APTUSDT/2024-01.parquet")
#df = pd.read_parquet("s3://tradebot-config-tokyo/data/v3/wf_stage0_nogate_2024-02.parquet#")
#df = pd.read_parquet("s3://tradebot-config-tokyo/data/v3/wf_stage0_gate_2024-02.parquet")
#df = pd.read_parquet("s3://tradebot-config-tokyo/data/bougie/BTCUSDT/BTCUSDT-1m-2024.parquet")
#df = pd.read_parquet("s3://tradebot-config-tokyo/data/v1/wf_stage0_tagged_2024-04.parquet")
#df = pd.read_parquet("s3://tradebot-config-tokyo/data/v1/wf_stage1_validated_2024-02.parquet")
df = pd.read_parquet("s3://tradebot-config-tokyo/data/v1_evalready/wf_stage0_evalready_2025-05.parquet")
print(df.columns)
print(df.head(3))

In [15]:
import pandas as pd
import numpy as np

from ml_bot.backtest.lib.io_s3 import read_parquet_s3

# ---------------------------------
# CONFIG
# ---------------------------------
RUNS = {
    "train_2024_04_to_2024_12": "ms_edge_05ea0ec3f9",
    "val_2025_01_to_2025_04":   "ms_edge_38fba815ae",
    "test_2025_05_to_2025_10":  "ms_edge_f12ec10933",
}

ROOT = "s3://tradebot-config-tokyo/research/ms_edge/runs"

BO_CFG = "BO | rw>-15 | gate=T60>0 | 60_if_neg_else_180"
MR_CFG = "MR | rw>-15 | gate=T60>0 | 60_if_neg_else_180"

# ---------------------------------
# LOAD + BUILD SUMMARY
# ---------------------------------
rows = []

for split_name, run_id in RUNS.items():
    run_root = f"{ROOT}/run_id={run_id}"

    pool = read_parquet_s3(f"{run_root}/bootstrap_pool.parquet")
    router = read_parquet_s3(f"{run_root}/router_summary.parquet")
    candidates = read_parquet_s3(f"{run_root}/router_candidates.parquet")

    # BO pool
    bo = pool.loc[pool["config"] == BO_CFG].copy()
    if len(bo):
        bo = bo.iloc[0]
        rows.append({
            "split": split_name,
            "run_id": run_id,
            "strategy": "BO_pool",
            "mr_buckets": "",
            "n_trades": int(bo["n_trades"]),
            "n_days": int(bo["n_days"]),
            "EV_bps": float(bo["EV_bps"]),
            "CI_blk_lo": float(bo["CI_blk_lo"]),
            "CI_blk_hi": float(bo["CI_blk_hi"]),
        })

    # MR pool
    mr = pool.loc[pool["config"] == MR_CFG].copy()
    if len(mr):
        mr = mr.iloc[0]
        rows.append({
            "split": split_name,
            "run_id": run_id,
            "strategy": "MR_pool",
            "mr_buckets": "",
            "n_trades": int(mr["n_trades"]),
            "n_days": int(mr["n_days"]),
            "EV_bps": float(mr["EV_bps"]),
            "CI_blk_lo": float(mr["CI_blk_lo"]),
            "CI_blk_hi": float(mr["CI_blk_hi"]),
        })

    # Current router
    if len(router):
        r = router.iloc[0]
        rows.append({
            "split": split_name,
            "run_id": run_id,
            "strategy": "ROUTER_current",
            "mr_buckets": "current_rule",
            "n_trades": int(r["n_trades"]),
            "n_days": int(r["n_days"]),
            "EV_bps": float(r["EV_bps"]),
            "CI_blk_lo": float(r["CI_blk_lo"]),
            "CI_blk_hi": float(r["CI_blk_hi"]),
        })

    # Best router candidate by CI lower bound, then EV
    cand = candidates.sort_values(["CI_blk_lo", "EV_bps"], ascending=[False, False]).copy()
    if len(cand):
        c = cand.iloc[0]
        rows.append({
            "split": split_name,
            "run_id": run_id,
            "strategy": "ROUTER_best_candidate",
            "mr_buckets": str(c["mr_buckets"]),
            "n_trades": int(c["n_trades"]),
            "n_days": int(c["n_days"]),
            "EV_bps": float(c["EV_bps"]),
            "CI_blk_lo": float(c["CI_blk_lo"]),
            "CI_blk_hi": float(c["CI_blk_hi"]),
        })

summary = pd.DataFrame(rows)

# nice ordering
strategy_order = {
    "BO_pool": 0,
    "MR_pool": 1,
    "ROUTER_current": 2,
    "ROUTER_best_candidate": 3,
}
split_order = {k: i for i, k in enumerate(RUNS.keys())}

summary["split_ord"] = summary["split"].map(split_order)
summary["strat_ord"] = summary["strategy"].map(strategy_order)

summary = (
    summary
    .sort_values(["split_ord", "strat_ord"])
    .drop(columns=["split_ord", "strat_ord"])
    .reset_index(drop=True)
)

print("\n=== COMPARATIVE SUMMARY (train / val / test) ===\n")
print(summary.to_string(index=False))


=== COMPARATIVE SUMMARY (train / val / test) ===

                   split             run_id              strategy   mr_buckets  n_trades  n_days   EV_bps  CI_blk_lo  CI_blk_hi
train_2024_04_to_2024_12 ms_edge_05ea0ec3f9               BO_pool                    581     141 6.777345   5.049527   8.620439
train_2024_04_to_2024_12 ms_edge_05ea0ec3f9               MR_pool                    433     126 6.540258   5.019656   8.138371
train_2024_04_to_2024_12 ms_edge_05ea0ec3f9        ROUTER_current current_rule       550     137 7.017701   5.097378   8.861508
train_2024_04_to_2024_12 ms_edge_05ea0ec3f9 ROUTER_best_candidate     b1,b2,b3       483     136 7.901201   6.194859   9.593327
  val_2025_01_to_2025_04 ms_edge_38fba815ae               BO_pool                    466      73 5.156399   3.827228   6.392125
  val_2025_01_to_2025_04 ms_edge_38fba815ae               MR_pool                    388      68 7.642358   4.997662  10.290608
  val_2025_01_to_2025_04 ms_edge_38fba815ae        RO

In [16]:
pivot_ev = summary.pivot_table(
    index=["strategy", "mr_buckets"],
    columns="split",
    values="EV_bps",
    aggfunc="first"
)

pivot_ci = summary.pivot_table(
    index=["strategy", "mr_buckets"],
    columns="split",
    values="CI_blk_lo",
    aggfunc="first"
)

print("\n=== EV_bps PIVOT ===\n")
print(pivot_ev.to_string())

print("\n=== CI_blk_lo PIVOT ===\n")
print(pivot_ci.to_string())


=== EV_bps PIVOT ===

split                               test_2025_05_to_2025_10  train_2024_04_to_2024_12  val_2025_01_to_2025_04
strategy              mr_buckets                                                                             
BO_pool                                            4.510294                  6.777345                5.156399
MR_pool                                            3.745035                  6.540258                7.642358
ROUTER_best_candidate b0,b1,b3,b4                       NaN                       NaN                7.469791
                      b1,b2,b3                          NaN                  7.901201                     NaN
                      b2,b3                        4.663435                       NaN                     NaN
ROUTER_current        current_rule                 4.786846                  7.017701                5.232878

=== CI_blk_lo PIVOT ===

split                               test_2025_05_to_2025_10  train_2024